# PangIA – Corpus Ingestion Notebook

This notebook runs `tools/ingest_corpus.py` to:

1. **Discover** all PDF, CSV, GeoJSON and JSON files inside a corpus folder.
2. **Analyse** them with an LLM to extract entities, relationships, geospatial data and textual knowledge.
3. **Generate** a PangIA seed-theme Python module that populates **Neo4j**, **PostGIS**, **RDF / GraphDB** and **ChromaDB**.
4. **Optionally** seed the live databases directly.

---

## Prerequisites

```bash
pip install langchain-openai langchain-core pdfplumber
```

Run this notebook from the **`backend/`** directory (or add it to `sys.path` in the cell below).

## 1 – Configuration

Edit the variables in this cell before running the notebook.

In [ ]:
import sys
import os
from pathlib import Path

# ── Path setup ────────────────────────────────────────────────────────────────
# Add the backend/ directory to sys.path so that both `tools` and `app`
# packages are importable.  Adjust if you run from a different working dir.
BACKEND_DIR = Path(".").resolve()          # backend/  (default)
# BACKEND_DIR = Path("../backend").resolve()  # if running from project root
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# ── Corpus ────────────────────────────────────────────────────────────────────
# Folder that contains the files to ingest.  Supported: .pdf .csv .json .geojson
# Override this to point at any directory on your machine.
CORPUS_PATH = BACKEND_DIR.parent / "corpus"   # <repo-root>/corpus  (default)
# CORPUS_PATH = Path("/data/my_corpus")        # absolute override example

# ── Theme ─────────────────────────────────────────────────────────────────────
# Short identifier used as the Python module name for the generated seed file.
# Must contain only letters, digits and underscores.
THEME_NAME = "custom"

# ── Output ────────────────────────────────────────────────────────────────────
# Where to write the generated seed file.
# Leave None to auto-place at backend/app/db/themes/<THEME_NAME>.py
OUTPUT_PATH = None
# OUTPUT_PATH = BACKEND_DIR / "app" / "db" / "themes" / f"{THEME_NAME}.py"

# ── LLM ───────────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")   # or paste your key here
OPENAI_MODEL   = "gpt-4o-mini"                      # or "gpt-4o", "gpt-4-turbo" …

# ── Token / context limits ────────────────────────────────────────────────────
# Maximum characters of corpus text forwarded to each LLM call.
# Increase for more capable models, decrease for smaller context windows.
MAX_CONTENT_CHARS = 16_000

print(f"Corpus  : {CORPUS_PATH}")
print(f"Theme   : {THEME_NAME}")
print(f"Model   : {OPENAI_MODEL}")
print(f"Backend : {BACKEND_DIR}")

## 2 – Discover corpus files

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")

from tools.ingest_corpus import CorpusIngestor

ingestor = CorpusIngestor(
    corpus_path=CORPUS_PATH,
    theme_name=THEME_NAME,
    openai_api_key=OPENAI_API_KEY,
    openai_model=OPENAI_MODEL,
    max_content_chars=MAX_CONTENT_CHARS,
)

files = ingestor.discover_files()
print(f"\nFound {len(files)} file(s):")
for f in files:
    print(f"  {f.relative_to(CORPUS_PATH)}  ({f.suffix})")

## 3 – Run ingestion

This step calls the LLM five times (Neo4j, PostGIS, RDF, ChromaDB, suggestions).
Execution time depends on corpus size and network latency.

In [ ]:
result = ingestor.ingest()

print("\n── Ingestion summary ─────────────────────────────────────────────")
print(f"Theme              : {result.theme_name}")
print(f"Files processed    : {len(result.files_processed)}")
print(f"Neo4j statements   : {len(result.neo4j_statements)}")
print(f"PostGIS statements : {len(result.postgis_statements)}")
print(f"RDF Turtle chars   : {len(result.graphdb_turtle)}")
print(f"Vector documents   : {len(result.chroma_documents)}")
print(f"UI suggestions     : {len(result.suggestions)}")

## 4 – Preview generated data

In [ ]:
print("── Neo4j schema prompt ───────────────────────────────────────────")
print(result.neo4j_schema_prompt or "(none)")

print("\n── First 3 Cypher statements ─────────────────────────────────────")
for stmt in result.neo4j_statements[:3]:
    print(stmt[:300], "\n")

In [ ]:
print("── PostGIS schema prompt ─────────────────────────────────────────")
print(result.postgis_schema_prompt or "(none)")

print("\n── First 3 SQL statements ────────────────────────────────────────")
for stmt in result.postgis_statements[:3]:
    print(stmt[:300], "\n")

In [ ]:
print("── RDF named graph ───────────────────────────────────────────────")
print(result.graphdb_named_graph)

print("\n── First 500 chars of Turtle ─────────────────────────────────────")
print(result.graphdb_turtle[:500] or "(none)")

In [ ]:
print("── First 3 vector documents ──────────────────────────────────────")
for doc in result.chroma_documents[:3]:
    print("  text    :", doc["text"][:120])
    print("  metadata:", doc["metadata"])
    print()

In [ ]:
print("── UI suggestions ────────────────────────────────────────────────")
for i, s in enumerate(result.suggestions, 1):
    print(f"  {i:2d}. {s}")

## 5 – Save seed file

Writes the generated theme to `backend/app/db/themes/<theme_name>.py`.
After saving, set `SEED_THEME=<theme_name>` in `.env` and restart the backend.

In [ ]:
seed_path = ingestor.save_seed_file(result, output_path=OUTPUT_PATH)

print(f"\n✓  Seed file written to: {seed_path}")
print(f"\nNext steps:")
print(f"  1. Review the generated file: {seed_path}")
print(f"  2. Add to .env:  SEED_DB=true  SEED_THEME={THEME_NAME}")
print(f"  3. Restart the backend to seed all databases.")

## 6 – (Optional) Preview generated seed file

In [ ]:
print(seed_path.read_text()[:3000])

## 7 – (Optional) Seed live databases directly

Requires the backend `.env` to be loaded and all databases to be running.

In [ ]:
import asyncio
from dotenv import load_dotenv

# Load .env from backend/
load_dotenv(BACKEND_DIR / ".env")

# Convert result to a live SeedTheme and seed all databases
theme = ingestor.to_seed_theme(result)

from app.db.seed import (
    _seed_neo4j,
    _seed_postgis,
    _seed_graphdb,
    _seed_chroma,
)

async def seed_all_live():
    await _seed_neo4j(theme)
    await _seed_postgis(theme)
    await _seed_graphdb(theme)
    await _seed_chroma(theme)
    print("All databases seeded successfully.")

await seed_all_live()